Business Problem
What will next month's sales be

so we are provide the necessary information which could help in 
inventory management 
budget planning
marketing spent
revenue forcasting 

In [ ]:
<h2h>Reading Gold Layer </h2h>

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum

spark=(
    SparkSession.builder
    .appName("Sales predection")
    .master("local[*]")
    .getOrCreate()
)
sales_df = spark.read.parquet("output/gold/sales_fact")
sales_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)



In [ ]:
<h2>Create Monthly revenue Dataset</h2>

In [15]:
monthly_sales_df=(
    sales_df
    .groupBy(
        "purchase_year",
        "purchase_month"
     )
    .agg(
        sum("payment_value").alias("monthly_revenue")
    )
    .orderBy("purchase_year","purchase_month")

)
monthly_sales_df.show()

+-------------+--------------+------------------+
|purchase_year|purchase_month|   monthly_revenue|
+-------------+--------------+------------------+
|         2016|             9|            347.52|
|         2016|            10|          73914.58|
|         2016|            12|             19.62|
|         2017|             1|187779.41000000006|
|         2017|             2|344134.78999999986|
|         2017|             3| 526961.6599999998|
|         2017|             4|505665.52999999997|
|         2017|             5| 724504.5499999997|
|         2017|             6| 600753.2699999997|
|         2017|             7| 737293.0799999995|
|         2017|             8| 870105.8999999996|
|         2017|             9|1015849.5699999997|
|         2017|            10|1021169.2699999993|
|         2017|            11| 1583869.009999999|
|         2017|            12|        1042855.86|
|         2018|             1| 1408365.649999999|
|         2018|             2| 1306048.799999999|


In [19]:
# converting pyspark to poandas since machine laearning library worked with panadas
import pandas as pd 
sales_pd=monthly_sales_df.toPandas()
sales_pd.head()

,purchase_year,purchase_month,monthly_revenue
0,2016,9,347.52
1,2016,10,73914.58
2,2016,12,19.62
3,2017,1,187779.41
4,2017,2,344134.79


In [ ]:
<h1> Creating time index </h1>

In [22]:
import pandas as pd

sales_pd["date"] = pd.to_datetime(
    sales_pd["purchase_year"].astype(str)
    + "-"
    + sales_pd["purchase_month"].astype(str)
    + "-01"
)
print(sales_pd)

    purchase_year  purchase_month  monthly_revenue       date
0            2016               9           347.52 2016-09-01
1            2016              10         73914.58 2016-10-01
2            2016              12            19.62 2016-12-01
3            2017               1        187779.41 2017-01-01
4            2017               2        344134.79 2017-02-01
5            2017               3        526961.66 2017-03-01
6            2017               4        505665.53 2017-04-01
7            2017               5        724504.55 2017-05-01
8            2017               6        600753.27 2017-06-01
9            2017               7        737293.08 2017-07-01
10           2017               8        870105.90 2017-08-01
11           2017               9       1015849.57 2017-09-01
12           2017              10       1021169.27 2017-10-01
13           2017              11       1583869.01 2017-11-01
14           2017              12       1042855.86 2017-12-01
15      

In [25]:
# creating simple forcasting model 

sales_pd["purchase_month"] =range(
    1,len(sales_pd)+1
)


In [ ]:
# train mode using Linear regeression 